# Notebook 13: Running Local Models

**Series:** LLM Systems & Inference  
**Prerequisites:** Notebooks 09 (Quantization), 12 (Serving Metrics)

---

## Background: The Local Inference Revolution

In early 2023, running a capable language model locally — on your own hardware, completely offline — was essentially impossible for anyone without a top-tier GPU cluster. The smallest usable model (GPT-2) was too weak to be useful. GPT-3 and its successors were API-only. The gap between "what I can run" and "what's actually useful" was enormous.

Three things changed everything in the span of a few months:

**1. The LLaMA leak (February 2023)**  
Meta released LLaMA weights to researchers, and they leaked publicly within days. For the first time, a genuinely capable model (LLaMA-7B performed comparably to GPT-3 on many benchmarks) was available to anyone. The community immediately started building.

**2. llama.cpp (March 2023)**  
Georgi Gerganov, a Bulgarian developer known for his whisper.cpp audio transcription project, ported LLaMA inference to pure C++ with no dependencies. Within 48 hours of the LLaMA leak, he had LLaMA running on a MacBook. Within a week, he'd added quantization (4-bit), and suddenly 7B models ran at 10-20 tokens/second on consumer hardware. The GitHub repo went from zero to 20,000 stars in a week.

**3. Apple Silicon (M1/M2/M3)**  
Apple's unified memory architecture turned out to be ideal for LLM inference. A MacBook Pro M3 Max with 128GB unified memory can hold a 70B model entirely in RAM and access it at ~400 GB/s — far better bandwidth per dollar than a discrete GPU setup of comparable memory capacity. The M-series chips became the premier platform for local LLM inference.

### The Three Runtimes

Today, three runtimes dominate local inference:

**llama.cpp**: The original. C++ with Metal/CUDA/CPU backends. Best compatibility, widest model support, GGUF quantization format. Powers most local inference under the hood.

**Ollama**: A polished wrapper around llama.cpp that adds a REST API, model management (pull/push/rm), and a clean CLI. Think of it as "Docker for LLMs" — you `ollama pull llama3` and it just works. Ideal for development and integration.

**MLX**: Apple's own framework for machine learning on Apple Silicon, released in late 2023. Uses Metal shaders and Apple's Accelerate framework for native M-series performance. Often 20-40% faster than llama.cpp on Apple Silicon for specific workloads. Best for maximum performance on Mac.

### Why This Matters

Local inference enables things that API-based inference fundamentally cannot:
- **Privacy**: Your data never leaves your machine
- **Cost**: No per-token charges for high-volume workloads
- **Latency**: No network round-trip; sub-10ms TTFT possible
- **Offline**: Works without internet; runs in air-gapped environments
- **Customization**: Fine-tune locally, serve locally, evaluate locally

---

## What You'll Build

This notebook is more hands-on than the others — you'll actually install and run models locally:
1. **Ollama setup and model management** — pull models, list, inspect
2. **Basic inference** — generate text, compare models
3. **Benchmarking** — measure TTFT, TBT, throughput with the benchpress harness
4. **Model size vs quality tradeoffs** — quantization levels on real outputs
5. **MLX comparison** — when to use MLX vs Ollama vs llama.cpp
6. **GGUF internals** — what's actually in the file format

> **Note:** This notebook requires Ollama to be installed (`brew install ollama`). Cells that require
> a running Ollama server are marked with `# REQUIRES OLLAMA`. If you don't have Ollama installed,
> you can still follow along — the benchmark simulation cells don't require it.

In [ ]:
!pip install -q requests numpy matplotlib

In [ ]:
import requests
import json
import time
import subprocess
import numpy as np
import matplotlib.pyplot as plt
from dataclasses import dataclass, field
from typing import Optional, Generator
import platform

# Detect Apple Silicon
is_apple_silicon = platform.processor() == 'arm' and platform.system() == 'Darwin'
print(f"Platform: {platform.system()} {platform.processor()}")
print(f"Apple Silicon: {is_apple_silicon}")
print(f"Python: {platform.python_version()}")

OLLAMA_URL = "http://localhost:11434"

def check_ollama() -> bool:
    """Check if Ollama server is running."""
    try:
        r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=2)
        return r.status_code == 200
    except Exception:
        return False

ollama_available = check_ollama()
print(f"\nOllama server: {'✅ running' if ollama_available else '❌ not running (simulation mode)'}")
if not ollama_available:
    print("  To install: brew install ollama && ollama serve")
    print("  Cells marked '# REQUIRES OLLAMA' will be skipped")

## Part 1: Ollama Model Management

Ollama uses a model registry similar to Docker Hub. Models are identified by name and tag,
pulled on first use, and cached locally.

In [ ]:
# REQUIRES OLLAMA
# Uncomment and run if Ollama is available

def list_ollama_models() -> list[dict]:
    """List locally available Ollama models."""
    r = requests.get(f"{OLLAMA_URL}/api/tags")
    return r.json().get('models', [])


def pull_model(model_name: str) -> None:
    """Pull a model from Ollama registry (streams progress)."""
    print(f"Pulling {model_name}...")
    r = requests.post(f"{OLLAMA_URL}/api/pull",
                      json={'name': model_name},
                      stream=True, timeout=600)
    for line in r.iter_lines():
        if line:
            data = json.loads(line)
            status = data.get('status', '')
            if 'total' in data and 'completed' in data:
                pct = data['completed'] / data['total'] * 100
                print(f"  {status}: {pct:.1f}%", end='\r')
            else:
                print(f"  {status}")
    print(f"\n{model_name} ready.")


def show_model_info(model_name: str) -> dict:
    """Get metadata about a local model."""
    r = requests.post(f"{OLLAMA_URL}/api/show", json={'name': model_name})
    return r.json()


if ollama_available:
    models = list_ollama_models()
    if models:
        print("Locally available models:")
        for m in models:
            size_gb = m.get('size', 0) / 1e9
            print(f"  {m['name']:<40} {size_gb:.1f} GB  (modified: {m.get('modified_at','')[:10]})")
    else:
        print("No models found. Pull one with: ollama pull llama3.2")
        print("Recommended for this notebook: llama3.2:1b, llama3.2:3b, llama3.1:8b")
else:
    print("(Ollama not available — showing example output)")
    example_models = [
        ('llama3.2:1b', 1.3),
        ('llama3.2:3b', 2.0),
        ('llama3.1:8b', 4.9),
        ('llama3.1:8b-instruct-q4_K_M', 4.9),
        ('mistral:7b', 4.1),
        ('codellama:7b', 4.1),
    ]
    print("Example models you might have:")
    for name, gb in example_models:
        print(f"  {name:<40} {gb:.1f} GB")

## Part 2: Basic Inference with Timing

Ollama exposes two endpoints:
- `/api/generate` — completion (raw text generation)
- `/api/chat` — chat format with message history

Both support streaming responses. We'll use streaming to measure TTFT and TBT precisely.

In [ ]:
@dataclass
class OllamaResult:
    model: str
    prompt: str
    response: str
    ttft_ms: float
    total_ms: float
    prompt_tokens: int
    output_tokens: int
    
    @property
    def tbt_ms(self) -> float:
        if self.output_tokens <= 1:
            return 0
        return (self.total_ms - self.ttft_ms) / (self.output_tokens - 1)
    
    @property
    def tokens_per_second(self) -> float:
        return self.output_tokens / (self.total_ms / 1000)
    
    def __str__(self):
        return (
            f"Model: {self.model}\n"
            f"Prompt tokens: {self.prompt_tokens}, Output tokens: {self.output_tokens}\n"
            f"TTFT: {self.ttft_ms:.0f}ms | TBT: {self.tbt_ms:.0f}ms | "
            f"Speed: {self.tokens_per_second:.1f} tok/s\n"
            f"Response: {self.response[:200]}{'...' if len(self.response) > 200 else ''}"
        )


def generate_with_timing(
    model: str,
    prompt: str,
    max_tokens: int = 200,
    temperature: float = 0.7,
) -> OllamaResult:
    """Generate text with precise TTFT and TBT measurement."""
    url = f"{OLLAMA_URL}/api/generate"
    payload = {
        'model': model,
        'prompt': prompt,
        'stream': True,
        'options': {
            'temperature': temperature,
            'num_predict': max_tokens,
        }
    }
    
    submit_time = time.perf_counter()
    first_token_time = None
    response_text = ''
    prompt_tokens = 0
    output_tokens = 0
    
    with requests.post(url, json=payload, stream=True, timeout=120) as resp:
        for line in resp.iter_lines():
            if not line:
                continue
            data = json.loads(line)
            
            token_text = data.get('response', '')
            if token_text and first_token_time is None:
                first_token_time = time.perf_counter()
            
            response_text += token_text
            output_tokens += 1 if token_text else 0
            
            if data.get('done', False):
                prompt_tokens = data.get('prompt_eval_count', 0)
                output_tokens = data.get('eval_count', output_tokens)
                break
    
    end_time = time.perf_counter()
    ttft = (first_token_time - submit_time) * 1000 if first_token_time else 0
    total = (end_time - submit_time) * 1000
    
    return OllamaResult(
        model=model, prompt=prompt, response=response_text.strip(),
        ttft_ms=ttft, total_ms=total,
        prompt_tokens=prompt_tokens, output_tokens=output_tokens
    )


# REQUIRES OLLAMA — run a test generation
TEST_MODEL = 'llama3.2:3b'  # Change to whatever model you have locally

if ollama_available:
    result = generate_with_timing(
        TEST_MODEL,
        "Explain attention mechanism in transformers in 3 sentences."
    )
    print(result)
else:
    print("(Ollama not available — simulated result:)")
    simulated = OllamaResult(
        model='llama3.2:3b',
        prompt='Explain attention mechanism in transformers in 3 sentences.',
        response='The attention mechanism allows a model to weigh the importance of different tokens...',
        ttft_ms=180, total_ms=4200, prompt_tokens=14, output_tokens=85
    )
    print(simulated)

## Part 3: Benchmarking Local Models

A proper benchmark runs multiple requests and reports percentile statistics.
Single-request measurements have high variance — you need at least 20-30 runs for
stable P99 estimates on TTFT.

In [ ]:
def benchmark_model(
    model: str,
    prompts: list[str],
    max_tokens: int = 100,
    n_warmup: int = 2,
) -> dict:
    """Benchmark a model with a set of prompts. Returns metric dict."""
    # Warmup
    print(f"  Warming up ({n_warmup} runs)...", end=' ', flush=True)
    for p in prompts[:n_warmup]:
        generate_with_timing(model, p, max_tokens)
    print("done")
    
    # Benchmark
    results = []
    print(f"  Benchmarking ({len(prompts)} runs):", end=' ', flush=True)
    for i, p in enumerate(prompts):
        r = generate_with_timing(model, p, max_tokens)
        results.append(r)
        print(f".", end='', flush=True)
    print(" done")
    
    ttfts = [r.ttft_ms for r in results]
    tbts = [r.tbt_ms for r in results if r.tbt_ms > 0]
    speeds = [r.tokens_per_second for r in results]
    
    return {
        'model': model,
        'n_runs': len(results),
        'ttft_p50': np.percentile(ttfts, 50),
        'ttft_p90': np.percentile(ttfts, 90),
        'ttft_p99': np.percentile(ttfts, 99),
        'tbt_p50': np.percentile(tbts, 50) if tbts else 0,
        'tbt_p90': np.percentile(tbts, 90) if tbts else 0,
        'speed_median': np.median(speeds),
        'speed_p10': np.percentile(speeds, 10),
        'raw_results': results,
    }


# Test prompts (diverse: short/long, factual/creative, code)
TEST_PROMPTS = [
    "What is the capital of France?",
    "Write a Python function to compute Fibonacci numbers.",
    "Explain gradient descent in simple terms.",
    "What are the key differences between TCP and UDP?",
    "Summarize the causes of World War I in 3 sentences.",
    "Write a SQL query to find the top 5 customers by revenue.",
    "What is attention in transformers?",
    "Describe the water cycle briefly.",
    "How does binary search work?",
    "What is a token in NLP?",
] * 3  # 30 runs total for stable estimates


if ollama_available:
    print(f"Benchmarking {TEST_MODEL}...")
    bench = benchmark_model(TEST_MODEL, TEST_PROMPTS, max_tokens=100)
    print(f"\nResults for {bench['model']}:")
    print(f"  Speed (median):  {bench['speed_median']:.1f} tok/s")
    print(f"  Speed (P10):     {bench['speed_p10']:.1f} tok/s")
    print(f"  TTFT P50:        {bench['ttft_p50']:.0f}ms")
    print(f"  TTFT P99:        {bench['ttft_p99']:.0f}ms")
    print(f"  TBT P50:         {bench['tbt_p50']:.0f}ms/token")
else:
    print("(Ollama not available — showing simulated benchmark results)")

## Part 4: Apple Silicon Performance Model

Even without Ollama running, we can reason about expected performance on Apple Silicon
using the same memory-bandwidth model from Notebook 12.

In [ ]:
# Apple Silicon specs
apple_chips = {
    'M1':         {'bw_gbs': 68.25,  'max_ram_gb': 16,   'year': 2020},
    'M1 Pro':     {'bw_gbs': 200,    'max_ram_gb': 32,   'year': 2021},
    'M1 Max':     {'bw_gbs': 400,    'max_ram_gb': 64,   'year': 2021},
    'M1 Ultra':   {'bw_gbs': 800,    'max_ram_gb': 128,  'year': 2022},
    'M2':         {'bw_gbs': 100,    'max_ram_gb': 24,   'year': 2022},
    'M2 Pro':     {'bw_gbs': 200,    'max_ram_gb': 32,   'year': 2023},
    'M2 Max':     {'bw_gbs': 400,    'max_ram_gb': 96,   'year': 2023},
    'M2 Ultra':   {'bw_gbs': 800,    'max_ram_gb': 192,  'year': 2023},
    'M3':         {'bw_gbs': 100,    'max_ram_gb': 24,   'year': 2023},
    'M3 Pro':     {'bw_gbs': 150,    'max_ram_gb': 36,   'year': 2023},
    'M3 Max':     {'bw_gbs': 300,    'max_ram_gb': 128,  'year': 2023},
    'M4':         {'bw_gbs': 120,    'max_ram_gb': 32,   'year': 2024},
    'M4 Pro':     {'bw_gbs': 273,    'max_ram_gb': 64,   'year': 2024},
    'M4 Max':     {'bw_gbs': 546,    'max_ram_gb': 128,  'year': 2024},
}

# Model sizes at different quantization levels
model_sizes = {
    'Llama-3.2-1B  Q4_K_M':  0.77,
    'Llama-3.2-3B  Q4_K_M':  2.0,
    'Llama-3.1-8B  Q4_K_M':  4.9,
    'Llama-3.1-8B  Q8_0':    8.5,
    'Llama-3.1-8B  FP16':    16.0,
    'Llama-3.1-70B Q4_K_M':  43.0,
    'Llama-3.1-70B Q8_0':    75.0,
    'Llama-3.1-70B FP16':    140.0,
}

def predicted_tps(model_gb: float, bandwidth_gbs: float, efficiency: float = 0.7) -> float:
    """Predicted tokens per second: bandwidth / model_size."""
    return (bandwidth_gbs * efficiency) / model_gb


# Print performance matrix for key chips and models
key_chips = ['M3 Max', 'M4 Pro', 'M4 Max', 'M2 Ultra']
key_models = [
    ('Llama-3.2-3B  Q4_K_M', 2.0),
    ('Llama-3.1-8B  Q4_K_M', 4.9),
    ('Llama-3.1-70B Q4_K_M', 43.0),
    ('Llama-3.1-70B FP16', 140.0),
]

print("Predicted tokens/sec on Apple Silicon (bandwidth model):")
print(f"{'Model':<28} {'RAM GB':>8} | ", end='')
for chip in key_chips:
    print(f"{chip:>12}", end='')
print()
print("-" * 90)

for model_name, model_gb in key_models:
    print(f"{model_name:<28} {model_gb:>8.0f} | ", end='')
    for chip in key_chips:
        bw = apple_chips[chip]['bw_gbs']
        max_ram = apple_chips[chip]['max_ram_gb']
        if model_gb > max_ram:
            print(f"{'OOM':>12}", end='')
        else:
            tps = predicted_tps(model_gb, bw)
            print(f"{tps:>11.0f}t", end='')
    print()

print("\n(Assumes 70% memory bandwidth efficiency, single-request decode)",)

In [ ]:
# Visualize: which model fits on which chip, and at what speed?
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: tokens/sec heatmap
selected_chips = ['M3 Pro', 'M3 Max', 'M4 Pro', 'M4 Max', 'M2 Ultra']
selected_models = list(model_sizes.items())

tps_grid = np.zeros((len(selected_models), len(selected_chips)))
for i, (mname, mgb) in enumerate(selected_models):
    for j, chip in enumerate(selected_chips):
        max_ram = apple_chips[chip]['max_ram_gb']
        bw = apple_chips[chip]['bw_gbs']
        if mgb <= max_ram:
            tps_grid[i, j] = predicted_tps(mgb, bw)
        else:
            tps_grid[i, j] = -1  # OOM

ax = axes[0]
masked = np.ma.masked_where(tps_grid < 0, tps_grid)
cmap = plt.cm.RdYlGn
cmap.set_bad(color='#bbbbbb')
im = ax.imshow(masked, aspect='auto', cmap=cmap, vmin=0, vmax=200)

for i in range(len(selected_models)):
    for j in range(len(selected_chips)):
        val = tps_grid[i, j]
        text = f"{val:.0f}t/s" if val >= 0 else "OOM"
        ax.text(j, i, text, ha='center', va='center', fontsize=8,
                color='black' if 0 < val < 150 else 'white')

ax.set_xticks(range(len(selected_chips)))
ax.set_xticklabels(selected_chips, rotation=20, ha='right')
ax.set_yticks(range(len(selected_models)))
ax.set_yticklabels([m for m, _ in selected_models], fontsize=8)
ax.set_title("Predicted Tokens/Sec on Apple Silicon", fontsize=11)
plt.colorbar(im, ax=ax, label="tokens/sec")

# Right: bandwidth comparison
ax2 = axes[1]
chip_names = list(apple_chips.keys())
bandwidths = [apple_chips[c]['bw_gbs'] for c in chip_names]
colors_bar = ['#3498db' if '4' in c else '#2ecc71' if '3' in c else '#e67e22' for c in chip_names]

bars = ax2.barh(chip_names, bandwidths, color=colors_bar, alpha=0.8, edgecolor='white')
ax2.set_xlabel("Memory Bandwidth (GB/s)")
ax2.set_title("Apple Silicon Memory Bandwidth")
ax2.axvline(200, color='gray', linestyle='--', alpha=0.5, label='200 GB/s reference')

# Add value labels
for bar, val in zip(bars, bandwidths):
    ax2.text(val + 5, bar.get_y() + bar.get_height()/2, f"{val} GB/s",
             va='center', fontsize=8)

# Legend for chip generations
legend_patches = [
    mpatches.Patch(color='#e67e22', label='M1/M2 series'),
    mpatches.Patch(color='#2ecc71', label='M3 series'),
    mpatches.Patch(color='#3498db', label='M4 series'),
]
import matplotlib.patches as mpatches
ax2.legend(handles=legend_patches, fontsize=8)
ax2.grid(alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig('apple_silicon_performance.png', dpi=120, bbox_inches='tight')
plt.show()

## Part 5: Runtime Comparison — Ollama vs MLX vs llama.cpp

The three main local inference runtimes each have different strengths.

In [ ]:
# Reference performance data from community benchmarks (M3 Max, 8B model)
# Source: various llm-benchmark and mlx community results, approximate values
runtime_comparison = [
    # (runtime, model, quant, tps_p50, ttft_ms, notes)
    ('llama.cpp (Metal)', 'Llama-3.1-8B', 'Q4_K_M', 82, 180, 'CLI tool, widely compatible'),
    ('Ollama (Metal)',    'Llama-3.1-8B', 'Q4_K_M', 78, 220, 'REST API, easy to use'),
    ('MLX',              'Llama-3.1-8B', 'Q4',     95, 150, 'Native Metal, Apple-optimized'),
    ('llama.cpp (Metal)', 'Llama-3.1-8B', 'Q8_0',  62, 190, ''),
    ('MLX',              'Llama-3.1-8B', 'FP16',   51, 160, ''),
    ('llama.cpp (Metal)', 'Llama-3.1-70B','Q4_K_M', 13, 900, 'Requires M3 Max 96GB+'),
    ('MLX',              'Llama-3.1-70B','Q4',     16, 750, ''),
]

print("Runtime Comparison (M3 Max, community benchmarks, approximate):")
print(f"{'Runtime':<22} {'Model':<18} {'Quant':<10} {'tok/s':>8} {'TTFT':>8}  Notes")
print("-" * 85)
for rt, model, quant, tps, ttft, notes in runtime_comparison:
    print(f"{rt:<22} {model:<18} {quant:<10} {tps:>8} {ttft:>7}ms  {notes}")

print("""
Key takeaways:
  • MLX is ~10-20% faster than llama.cpp on Apple Silicon (native Metal shaders)
  • Ollama wraps llama.cpp — nearly identical performance, much easier to use
  • Q4_K_M is the sweet spot: best quality-per-GB for most use cases
  • TTFT is dominated by KV memory allocation, not model loading (models stay hot)
  • 70B models need 96GB+ unified memory (M3 Max maxed out, or M2/M4 Ultra)
""")

## Part 6: GGUF Format Internals

GGUF ("GPT-Generated Unified Format") is the file format used by llama.cpp and Ollama.
Understanding what's in the file helps you choose the right variant and diagnose issues.

In [ ]:
# GGUF naming convention decoder
def decode_gguf_name(filename: str) -> dict:
    """
    Parse a GGUF filename into its components.
    Format: ModelName-QuantType-[K_][S/M/L].gguf
    
    Quantization types:
    - Q4_K_M: 4-bit, K-quant (asymmetric), Medium quality
    - Q8_0:   8-bit, symmetrically quantized
    - F16:    16-bit float (no quantization)
    - IQ4_XS: I-quant 4-bit extra small (newer, better quality)
    """
    import re
    result = {'filename': filename}
    
    # Extract quant type
    quant_match = re.search(r'(IQ[0-9]_\w+|Q[0-9]+_[\w]+|F16|F32|BF16)', filename, re.IGNORECASE)
    if quant_match:
        result['quant_type'] = quant_match.group(1).upper()
    
    # Size variant
    size_match = re.search(r'_(XS|S|M|L|XL)\.gguf', filename, re.IGNORECASE)
    if size_match:
        size_map = {'XS': 'Extra Small', 'S': 'Small', 'M': 'Medium', 'L': 'Large', 'XL': 'Extra Large'}
        result['quality_tier'] = size_map.get(size_match.group(1).upper(), size_match.group(1))
    
    return result


# GGUF quantization guide
quant_guide = [
    # (type, bits_eff, size_factor, quality, use_case)
    ('Q2_K',   2.5, 0.31, 'Poor',      'Absolute minimum RAM; not recommended for quality-sensitive tasks'),
    ('Q3_K_S', 3.0, 0.38, 'Low',       'Very tight RAM constraints'),
    ('Q3_K_M', 3.3, 0.41, 'Moderate',  'Small RAM, acceptable quality'),
    ('Q4_K_S', 4.0, 0.50, 'Good',      'Balanced; missing some nuance vs M'),
    ('Q4_K_M', 4.5, 0.56, 'Very Good', '⭐ Best default choice — sweet spot'),
    ('Q5_K_M', 5.5, 0.69, 'Excellent', 'Near-lossless; use when RAM allows'),
    ('Q6_K',   6.6, 0.83, 'Near-FP16', 'Minimal quality loss vs FP16'),
    ('Q8_0',   8.0, 1.00, '≈FP16',     'Effectively lossless; FP16 reference'),
    ('F16',    16.0, 2.0, 'Exact',      'Full precision; 2x RAM vs Q8, minimal gain'),
    ('IQ4_XS', 4.3, 0.54, 'Very Good', 'Newer i-quant; slightly better than Q4_K_M'),
    ('IQ3_M',  3.7, 0.46, 'Good',      'Newer i-quant; better than Q3_K_M'),
]

print("GGUF Quantization Reference Guide:")
print(f"{'Type':<12} {'Eff Bits':>9} {'Size Factor':>12} {'Quality':<12} Use Case")
print("-" * 90)
for qtype, bits, factor, quality, use_case in quant_guide:
    print(f"{qtype:<12} {bits:>9.1f} {factor:>12.2f} {quality:<12} {use_case}")

print("\nFor a 7B model (~14GB FP16):")
fp16_gb = 14.0
for qtype, bits, factor, quality, _ in quant_guide:
    gb = fp16_gb * factor
    tps_7b = predicted_tps(gb, 300)  # M3 Max bandwidth
    print(f"  {qtype:<12}: {gb:.1f} GB → ~{tps_7b:.0f} tok/s on M3 Max")

## Part 7: Choosing Your Runtime — Decision Framework

In [ ]:
decision_tree = """
Local LLM Runtime Selection Guide
==================================

Q: What's your primary goal?

  → "I want to integrate LLM into my app via API"
     Use: Ollama
     Why: REST API at localhost:11434, OpenAI-compatible endpoint,
          auto model management, Docker-style simplicity
     How: `ollama serve` → call /api/generate or /api/chat

  → "I want maximum speed on Apple Silicon"
     Use: MLX (mlx-lm package)
     Why: Native Metal shaders, 10-20% faster than llama.cpp on M-series
     How: `pip install mlx-lm` → `mlx_lm.generate --model ...`
     Note: Only works on Apple Silicon; models need to be in MLX format

  → "I want maximum hardware compatibility (CUDA/CPU/Metal)"
     Use: llama.cpp directly
     Why: Works on everything — NVIDIA, AMD, Apple Silicon, CPU-only
          Best for embedded/edge/non-Mac environments
     How: Build from source or use llama-cpp-python

  → "I want Python integration with Hugging Face models"
     Use: transformers + PEFT (for fine-tuning) or bitsandbytes (quantization)
     Why: Best ecosystem, fine-tuning support, widest model selection
     Note: Slower than llama.cpp for pure inference

  → "I'm on Linux/Windows with NVIDIA GPU"
     Use: vLLM (for serving) or llama.cpp with CUDA backend (for personal use)
     Why: vLLM has the best throughput; llama.cpp is simpler to set up

Model Size Decision:
  RAM ≤ 8GB:   Use 3B models at Q4_K_M (e.g., Llama-3.2-3B)
  RAM 16-24GB: Use 8B models at Q4_K_M (e.g., Llama-3.1-8B)
  RAM 32-64GB: Use 8B at F16 OR 34B at Q4_K_M
  RAM 96-128GB: Use 70B at Q4_K_M — best local quality available
"""

print(decision_tree)

## Summary

Local inference went from impossible to routine in under 18 months, driven by the LLaMA
leak, llama.cpp, and Apple Silicon's unified memory architecture.

### Key Takeaways

**The Stack**  
- **llama.cpp**: C++ runtime, GGUF format, works everywhere (CUDA/Metal/CPU)
- **Ollama**: Docker-for-LLMs wrapper around llama.cpp; REST API + model management
- **MLX**: Apple's native framework; 10-20% faster on M-series; Mac only

**Why Apple Silicon Wins for Local Inference**  
Unified memory eliminates the CPU↔GPU copy bottleneck. An M3 Max with 128GB unified
memory has ~300 GB/s bandwidth accessible to both CPU and GPU cores simultaneously.
A comparable discrete GPU (e.g., RTX 4090) has only 24GB — can't even fit a 70B model.

**Quantization for Local Inference**  
Q4_K_M is the default recommendation: great quality, half the size of FP16, and
2× the tokens/second on bandwidth-limited hardware. For production quality: Q5_K_M
or Q6_K at the cost of more RAM.

**Measuring Performance**  
Never report a single timing number. Measure TTFT and TBT separately across 20+ runs,
report P50 and P99, include confidence intervals. Token/second alone doesn't tell the
full story — TTFT matters more for interactive use cases.

### What's Next (Notebook 14: vLLM Internals)

For production server-side inference, vLLM is the dominant open-source solution. Notebook 14
walks through how it works internally: how PagedAttention (notebook 07) and continuous batching
(notebook 11) combine, what the scheduler looks like in real code, and how to configure it
for different serving workloads.